# Sudoku como CSP: fuerza bruta frente a propagación

**Facsímil 2 · Inteligencia clásica** — capítulos 5 a 7
(SAT y CSP; variables, dominios y restricciones; propagación, *backtracking* y heurísticas).

Un sudoku parece un pasatiempo, pero es la puerta de entrada a una de las familias de problemas más
útiles de la informática: los **problemas de satisfacción de restricciones** (CSP). Horarios,
asignación de aulas, configuración de productos, planificación de turnos... todos son CSP por debajo.
En este cuaderno resuelves un sudoku de dos maneras —a lo bruto y con cabeza— y **cuentas el esfuerzo
de cada una**. La diferencia es enorme, y la razón de esa diferencia —*propagar* las consecuencias de
cada decisión antes de seguir— es una idea que reaparece en agentes, planificadores y verificadores.

### Qué vas a aprender
- Qué es un **CSP**: variables, dominios y restricciones, con el sudoku como ejemplo concreto.
- A **ver** el problema: dibujar el tablero y un mapa de calor de cuántas opciones tiene cada casilla.
- Cómo funciona el **backtracking** (prueba y retrocede) y por qué, a secas, es muy lento.
- Qué es la **propagación de restricciones** (*forward checking*), la heurística **MRV** (variable
  más restringida primero) y los *naked singles*, y cuánto ahorran.
- A **medir cada ingrediente por separado** (ablación) y a ver que el mismo esqueleto colorea un mapa.
- A medir, con números reales, por qué «pensar antes de probar» convierte miles de tanteos en decenas.

### Cuánto cuesta
Unos 15 minutos. CPU, solo Python estándar (y matplotlib para los dibujos).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# Solo Python estandar.
# 0 = casilla vacia. Un sudoku de dificultad media.
TABLERO = [
    [5,3,0, 0,7,0, 0,0,0],
    [6,0,0, 1,9,5, 0,0,0],
    [0,9,8, 0,0,0, 0,6,0],
    [8,0,0, 0,6,0, 0,0,3],
    [4,0,0, 8,0,3, 0,0,1],
    [7,0,0, 0,2,0, 0,0,6],
    [0,6,0, 0,0,0, 2,8,0],
    [0,0,0, 4,1,9, 0,0,5],
    [0,0,0, 0,8,0, 0,7,9],
]
def pinta(g):
    for i, fila in enumerate(g):
        if i % 3 == 0 and i: print("------+-------+------")
        print(" ".join((str(v) if v else ".") + (" |" if j in (2,5) else "")
                        for j, v in enumerate(fila)))
pinta(TABLERO)


## 1. Un sudoku es un CSP

Un **problema de satisfacción de restricciones** tiene tres ingredientes:

- **Variables:** las cosas a decidir. Aquí, las 81 casillas.
- **Dominios:** los valores posibles de cada variable. Aquí, los números del 1 al 9 (o el valor fijo,
  si la casilla viene dada).
- **Restricciones:** las reglas que limitan qué combinaciones son válidas. Aquí: en cada fila, columna
  y caja de 3×3 no se puede repetir ningún número.

Resolver el CSP es asignar un valor a cada variable de modo que **todas** las restricciones se
cumplan. Esta forma de plantear el problema es poderosísima: el *mismo* método que resuelve el sudoku
sirve para asignar horarios o configurar una máquina, solo cambiando variables, dominios y reglas.
Empezamos por codificar las restricciones en una sola función.


In [ ]:
def valido(g, fila, col, v):
    if v in g[fila]: return False                       # restriccion de fila
    if v in [g[i][col] for i in range(9)]: return False # restriccion de columna
    f0, c0 = 3*(fila//3), 3*(col//3)
    for i in range(f0, f0+3):
        for j in range(c0, c0+3):
            if g[i][j] == v: return False               # restriccion de caja
    return True
print("Las tres restricciones del sudoku, en una funcion. Todo lo demas se construye encima.")


## 2. ¿Cómo de grande es el problema?

Antes de resolver nada, conviene hacerse una idea del tamaño del espacio de búsqueda. Si probáramos
combinaciones a ciegas, el número de tableros candidatos es **astronómico**: ningún ordenador puede
enumerarlos. Por eso la fuerza bruta pura no es una opción, y por eso hace falta buscar con cabeza.


In [ ]:
vacias = sum(1 for f in range(9) for c in range(9) if TABLERO[f][c] == 0)
ingenuo = 9 ** vacias                       # 9 valores por casilla, a lo loco
compatibles = 1                             # solo valores que respetan las pistas de salida
for f in range(9):
    for c in range(9):
        if TABLERO[f][c] == 0:
            compatibles *= max(1, sum(1 for v in range(1,10) if valido(TABLERO,f,c,v)))
print(f"Casillas vacias: {vacias}")
print(f"A ciegas (9 por casilla):        9^{vacias} ~ {float(ingenuo):.2e} combinaciones")
print(f"Solo valores compatibles:        ~ {float(compatibles):.2e} combinaciones")
print("\nAun mirando solo lo compatible, sigue siendo inabarcable: no se puede enumerar.")


## 3. Ver el problema: el tablero dibujado

Trabajar con listas de listas está bien para el ordenador, pero nosotros entendemos mejor un dibujo.
Pintamos la rejilla con sus cajas de 3×3 y las pistas iniciales en negro. Más adelante reutilizaremos
esta misma función para mostrar la solución (en gris, para distinguirla de las pistas).


In [ ]:
import matplotlib.pyplot as plt

def dibuja_tablero(g, pistas=None, titulo=""):
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    for k in range(10):                                  # rejilla: lineas gruesas cada 3
        ancho = 2.2 if k % 3 == 0 else 0.6
        ax.plot([0,9], [k,k], color="black", lw=ancho)
        ax.plot([k,k], [0,9], color="black", lw=ancho)
    for f in range(9):
        for c in range(9):
            v = g[f][c]
            if v:
                es_pista = (pistas is None) or (pistas[f][c] != 0)
                ax.text(c+0.5, 8.5-f, str(v), ha="center", va="center", fontsize=14,
                        color="black" if es_pista else "0.55",
                        fontweight="bold" if es_pista else "normal")
    ax.set_xlim(0,9); ax.set_ylim(0,9); ax.set_aspect("equal")
    ax.axis("off"); ax.set_title(titulo)
    plt.tight_layout(); plt.show()

dibuja_tablero(TABLERO, titulo="Las pistas de partida")


## 4. Mapa de calor: ¿por dónde conviene empezar?

No todas las casillas vacías son igual de difíciles. Algunas, con las pistas de salida, ya solo admiten
**una o dos opciones**; otras admiten muchas. Si pintamos el número de opciones de cada casilla, vemos
de un vistazo dónde está el problema más «apretado». Esa intuición es justo la de la heurística **MRV**
que usaremos luego: atacar primero la casilla con menos opciones, donde antes salta una contradicción.


In [ ]:
import numpy as np

opciones = np.zeros((9,9))
for f in range(9):
    for c in range(9):
        if TABLERO[f][c] == 0:
            opciones[f][c] = sum(1 for v in range(1,10) if valido(TABLERO,f,c,v))
        else:
            opciones[f][c] = np.nan                       # casilla ya fija: sin color

fig, ax = plt.subplots(figsize=(4.8, 4.3))
im = ax.imshow(opciones, cmap="YlOrRd")
for f in range(9):
    for c in range(9):
        if TABLERO[f][c]:
            ax.text(c, f, "·", ha="center", va="center", color="0.4")
        else:
            ax.text(c, f, int(opciones[f][c]), ha="center", va="center", fontsize=9)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Numero de opciones por casilla")
fig.colorbar(im, ax=ax, shrink=0.8, label="opciones")
plt.tight_layout(); plt.show()
print("Las casillas con 1-2 opciones son las mas restringidas: por ahi conviene empezar.")


## 5. Forma 1: backtracking a lo bruto

La estrategia ingenua, pero que **siempre funciona**: ve a la primera casilla vacía, prueba 1, 2, 3...
Si un valor es válido (no rompe ninguna restricción *ahora mismo*), colócalo y sigue con la siguiente
casilla. Si te atascas (ningún valor vale), **retrocede** (*backtrack*) y cambia la decisión anterior.
Es una búsqueda en profundidad sobre el espacio de asignaciones.

El problema: prueba un valor pensando solo en el presente, sin anticipar el desastre que provoca tres
casillas más adelante. Por eso explora muchísimo. Contamos cuántas veces coloca un número.


In [ ]:
def resolver_bruto(g, limite=None):
    intentos = [0]
    def backtrack(g):
        if limite is not None and intentos[0] >= limite:
            return False        # nos rendimos: este metodo no escala en los dificiles
        for f in range(9):
            for c in range(9):
                if g[f][c] == 0:
                    for v in range(1, 10):
                        if valido(g, f, c, v):
                            g[f][c] = v; intentos[0] += 1
                            if backtrack(g): return True
                            g[f][c] = 0          # deshacer y probar otro
                    return False                 # ningun valor sirve -> backtrack
        return True
    sol = [fila[:] for fila in g]
    backtrack(sol)
    return sol, intentos[0]

sol1, intentos_bruto = resolver_bruto(TABLERO)
print(f"Backtracking a lo bruto: {intentos_bruto} valores colocados hasta resolverlo.")


## 6. *Naked singles*: a veces basta con deducir

Antes de adivinar nada, hay una jugada gratis: si una casilla vacía solo admite **un** valor posible
(un *naked single*, «único desnudo»), ese valor es forzoso, lo colocamos sin más. Y al colocarlo,
puede que otra casilla quede también con una sola opción. Repetimos hasta que no haya más forzados.
Esto es **propagación pura**, sin búsqueda ni conjeturas. Veamos cuánto resuelve por sí sola.


In [ ]:
def propaga_singletons(g):
    sol = [fila[:] for fila in g]
    rellenadas = 0
    cambio = True
    while cambio:                                    # repetimos mientras haya forzados
        cambio = False
        for f in range(9):
            for c in range(9):
                if sol[f][c] == 0:
                    cand = [v for v in range(1,10) if valido(sol,f,c,v)]
                    if len(cand) == 1:               # una sola opcion: forzosa
                        sol[f][c] = cand[0]; rellenadas += 1; cambio = True
    return sol, rellenadas

vac0 = sum(1 for f in range(9) for c in range(9) if TABLERO[f][c] == 0)
solp, rellenadas = propaga_singletons(TABLERO)
quedan = sum(1 for f in range(9) for c in range(9) if solp[f][c] == 0)
print(f"Casillas vacias al empezar: {vac0}")
print(f"Rellenadas SOLO deduciendo (naked singles): {rellenadas}")
print(f"Quedan por decidir: {quedan}")
if quedan == 0:
    print("\nEste sudoku medio se resuelve ENTERO sin una sola conjetura: pura deduccion.")


## 7. Forma 2: propagar antes de probar (el método general)

Los *naked singles* resuelven los sudokus fáciles, pero en los difíciles llega un punto en que ninguna
casilla está forzada y hay que **conjeturar**. El método general combina deducir y conjeturar, con dos
mejoras clásicas de los CSP:

- ***Forward checking*** (propagación): cada casilla mantiene su **dominio** (qué valores le quedan). Al
  colocar un número, lo **tachamos** de los dominios de sus vecinas (fila, columna y caja). Si una
  vecina se queda sin opciones, sabemos *de inmediato* que esta rama es un callejón sin tener que llegar
  hasta ella.
- **Heurística MRV** (*minimum remaining values*): en vez de ir en orden, elegimos siempre la casilla
  con **menos opciones**. Es donde antes se descubre una contradicción, así que poda el árbol cuanto
  antes. (Es la intuición del mapa de calor de antes.)

Menos adivinar, más deducir.


In [ ]:
def resolver_propagando(g):
    intentos = [0]
    dom = {}
    for f in range(9):
        for c in range(9):
            dom[(f,c)] = {g[f][c]} if g[f][c] else {v for v in range(1,10) if valido(g,f,c,v)}
    sol = [fila[:] for fila in g]

    def quita(dom, f, c, v):
        cambios = []
        vecinas = set((f,j) for j in range(9)) | set((i,c) for i in range(9))
        f0, c0 = 3*(f//3), 3*(c//3)
        vecinas |= set((i,j) for i in range(f0,f0+3) for j in range(c0,c0+3))
        for (i,j) in vecinas:
            if (i,j) != (f,c) and sol[i][j] == 0 and v in dom[(i,j)]:
                dom[(i,j)] = dom[(i,j)] - {v}; cambios.append((i,j))
        return cambios

    def backtrack():
        vacias = [(f,c) for f in range(9) for c in range(9) if sol[f][c] == 0]
        if not vacias: return True
        f, c = min(vacias, key=lambda p: len(dom[p]))   # MRV: la mas restringida primero
        for v in sorted(dom[(f,c)]):
            if valido(sol, f, c, v):
                sol[f][c] = v; intentos[0] += 1
                guardado = dom[(f,c)]; dom[(f,c)] = {v}
                cambios = quita(dom, f, c, v)           # forward checking
                if all(len(dom[p]) > 0 for p in vacias if sol[p[0]][p[1]] == 0):
                    if backtrack(): return True
                for (i,j) in cambios: dom[(i,j)].add(v) # deshacer
                dom[(f,c)] = guardado; sol[f][c] = 0
        return False

    backtrack()
    return sol, intentos[0]

sol2, intentos_prop = resolver_propagando(TABLERO)
print(f"Con propagacion + MRV: {intentos_prop} valores colocados.\n")
pinta(sol2)


La solución, ahora dibujada: en **negro** las pistas de partida, en **gris** lo que ha deducido el
solucionador. Mismo tablero de antes, ya completo.


In [ ]:
dibuja_tablero(sol2, pistas=TABLERO, titulo="Resuelto (pistas en negro, deduccion en gris)")


## 8. El veredicto

Mismo problema, misma respuesta, dos formas de llegar. Compara el esfuerzo:


In [ ]:
print(f"A lo bruto:        {intentos_bruto:>5} colocaciones")
print(f"Propagando + MRV:  {intentos_prop:>5} colocaciones")
ahorro = 100 * (1 - intentos_prop / intentos_bruto)
print(f"\nPropagar te ahorra el {ahorro:.0f}% del trabajo. Misma solucion, una fraccion del esfuerzo.")
print("No es que el ordenador sea mas rapido: es que piensa antes de probar.")


**Por qué la diferencia es tan grande.** El backtracking bruto descubre los conflictos *tarde*,
cuando ya ha construido medio tablero sobre una base imposible, y tiene que deshacerlo todo. La
propagación los descubre *temprano*: en cuanto una casilla se queda sin opciones, poda esa rama
entera. Y la heurística MRV ataca justo por donde el problema está más apretado, donde antes salta la
contradicción. Combinadas, convierten una búsqueda enorme en una manejable.


## 9. Ablación: ¿cuánto aporta cada idea?

Hemos metido dos ideas a la vez (*forward checking* y MRV). ¿Cuánto pesa cada una? Lo limpio es una
**ablación**: el mismo solucionador con interruptores, probando las cuatro combinaciones. Escribimos
un único motor `resolver_csp(usar_mrv, usar_fc)`. Con los dos interruptores apagados reproduce el
método bruto; con los dos encendidos, el método propagado. Así la comparación es justa: solo cambian
las dos ideas que estudiamos.


In [ ]:
def resolver_csp(g, usar_mrv=False, usar_fc=False, limite=None):
    intentos = [0]
    sol = [fila[:] for fila in g]
    dom = None
    if usar_fc:                                      # los dominios solo se mantienen con forward checking
        dom = {}
        for f in range(9):
            for c in range(9):
                dom[(f,c)] = {g[f][c]} if g[f][c] else {v for v in range(1,10) if valido(g,f,c,v)}

    def opciones_de(f, c):                           # candidatos actuales de una casilla
        if usar_fc: return sorted(dom[(f,c)])
        return [v for v in range(1,10) if valido(sol,f,c,v)]

    def tam(f, c):                                   # cuantas opciones tiene (para MRV)
        if usar_fc: return len(dom[(f,c)])
        return sum(1 for v in range(1,10) if valido(sol,f,c,v))

    def quita(f, c, v):
        cambios = []
        vecinas = set((f,j) for j in range(9)) | set((i,c) for i in range(9))
        f0, c0 = 3*(f//3), 3*(c//3)
        vecinas |= set((i,j) for i in range(f0,f0+3) for j in range(c0,c0+3))
        for (i,j) in vecinas:
            if (i,j) != (f,c) and sol[i][j] == 0 and v in dom[(i,j)]:
                dom[(i,j)] = dom[(i,j)] - {v}; cambios.append((i,j))
        return cambios

    def backtrack():
        if limite is not None and intentos[0] >= limite: return False
        vacias = [(f,c) for f in range(9) for c in range(9) if sol[f][c] == 0]
        if not vacias: return True
        f, c = min(vacias, key=lambda p: tam(*p)) if usar_mrv else vacias[0]
        for v in opciones_de(f, c):
            if valido(sol, f, c, v):
                sol[f][c] = v; intentos[0] += 1
                if usar_fc:
                    guardado = dom[(f,c)]; dom[(f,c)] = {v}
                    cambios = quita(f, c, v)
                    if all(len(dom[p]) > 0 for p in vacias if sol[p[0]][p[1]] == 0):
                        if backtrack(): return True
                    for (i,j) in cambios: dom[(i,j)].add(v)
                    dom[(f,c)] = guardado
                else:
                    if backtrack(): return True
                sol[f][c] = 0
        return False

    backtrack()
    return sol, intentos[0]

# Comprobacion: con los interruptores al extremo, reproduce los dos metodos anteriores.
_, c_bruto = resolver_csp(TABLERO, usar_mrv=False, usar_fc=False)
_, c_full  = resolver_csp(TABLERO, usar_mrv=True,  usar_fc=True)
print(f"Sin MRV, sin FC = {c_bruto}  (bruto de antes: {intentos_bruto}) -> coincide: {c_bruto==intentos_bruto}")
print(f"Con MRV y FC    = {c_full}  (propagando de antes: {intentos_prop}) -> coincide: {c_full==intentos_prop}")


In [ ]:
combinaciones = [
    ("nada (bruto)",   False, False),
    ("solo FC",        False, True),
    ("solo MRV",       True,  False),
    ("FC + MRV",       True,  True),
]
print(f"{'metodo':<16}{'colocaciones':>14}")
print("-"*30)
resultados = {}
for nombre, mrv, fc in combinaciones:
    _, n = resolver_csp(TABLERO, usar_mrv=mrv, usar_fc=fc)
    resultados[nombre] = n
    print(f"{nombre:<16}{n:>14,}")
mejor = min(resultados.values()); peor = max(resultados.values())
print(f"\nDel peor al mejor: factor x{peor/mejor:.0f}. Cada idea recorta; juntas dan el minimo.")


**Lectura de la ablación.** Apagar las dos ideas nos devuelve al método bruto, el que más coloca.
Encender *cualquiera* de las dos ya recorta el esfuerzo, y encenderlas a la vez da el mínimo. No hay una
sola «idea mágica»: es la combinación de **propagar** las consecuencias y **elegir bien** la siguiente
casilla lo que hace manejable el problema. Esta forma de medir —quitar un ingrediente y ver cuánto se
nota— es la manera honesta de saber qué aporta cada cosa, en CSP y en casi cualquier sistema.


## 10. Cuanto más difícil, más se nota

La ventaja de propagar no es constante: **crece** con la dificultad. Cuantas menos pistas tiene el
sudoku, más caminos sin salida hay, y más sufre el método bruto (que los recorre todos) frente al
propagado (que los poda). Lo medimos con un sudoku conocido por ser duro para la fuerza bruta: empieza
casi vacío.


In [ ]:
DIFICIL = [
    [0,0,0, 0,0,0, 0,0,0],
    [0,0,0, 0,0,3, 0,8,5],
    [0,0,1, 0,2,0, 0,0,0],
    [0,0,0, 5,0,7, 0,0,0],
    [0,0,4, 0,0,0, 1,0,0],
    [0,9,0, 0,0,0, 0,0,0],
    [5,0,0, 0,0,0, 0,7,3],
    [0,0,2, 0,1,0, 0,0,0],
    [0,0,0, 0,4,0, 0,0,9],
]
LIMITE = 200_000   # presupuesto de intentos para el metodo bruto
_, bruto_d = resolver_bruto(DIFICIL, limite=LIMITE)
_, prop_d  = resolver_propagando(DIFICIL)
print(f"Sudoku medio   -> bruto: {intentos_bruto:>9,} colocaciones | propagando: {intentos_prop:>6,}")
if bruto_d >= LIMITE:
    print(f"Sudoku DIFICIL -> bruto: se rinde tras {LIMITE:>9,} SIN resolverlo | propagando: {prop_d:>6,} (RESUELTO)")
    print(f"\nEn los sudokus duros, el metodo bruto ni siquiera termina dentro de un limite razonable.")
    print("Propagar sigue siendo viable: la ventaja no es lineal, se dispara con la dificultad.")
else:
    print(f"Sudoku DIFICIL -> bruto: {bruto_d:>9,} | propagando: {prop_d:>6,} (factor x{bruto_d//max(prop_d,1)})")


## 11. Estable frente a impredecible

Otra forma de ver la diferencia: tomamos la solución del sudoku medio y le vamos quitando pistas (cada
vez más huecos), midiendo cuánto cuesta resolver cada versión. Lo revelador no es solo el tamaño, sino
la **regularidad**: propagar coloca casi tantos valores como casillas vacías hay (apenas retrocede),
sea cual sea el caso. El método bruto es una lotería: a veces tiene suerte, a veces se dispara a
decenas de miles de tanteos con el mismo número de huecos.


In [ ]:
def quita_pistas(resuelto, n):
    g = [fila[:] for fila in resuelto]
    cont = 0
    for f in range(9):
        for c in range(9):
            if cont < n:
                g[f][c] = 0; cont += 1
    return g

print(f"{'huecos':>7} | {'bruto (colocaciones)':>22} | {'propagando':>12}")
print("-"*48)
for n in [20, 30, 40, 50, 55]:
    p = quita_pistas(sol2, n)
    _, b = resolver_bruto(p, limite=120_000)
    _, q = resolver_propagando(p)
    bstr = f">={120_000:,} (no acaba)" if b >= 120_000 else f"{b:,}"
    print(f"{n:>7} | {bstr:>22} | {q:>12,}")
print("\nPropagar es estable y predecible; el bruto, una ruleta que a veces explota.")


## 12. Detectar lo imposible cuanto antes

Un buen método no solo resuelve rápido lo que tiene solución: también **descarta rápido** lo que no la
tiene. Metemos un conflicto en el sudoku medio (un 8 repetido en la misma fila) y vemos cuánto tarda
cada método en concluir que **no hay solución**.


In [ ]:
IMPOSIBLE = [fila[:] for fila in TABLERO]
IMPOSIBLE[3][1] = 8                 # la fila 3 ya tiene un 8: ahora hay dos -> sin solucion
_, n_bruto = resolver_bruto(IMPOSIBLE, limite=200_000)
_, n_prop  = resolver_propagando(IMPOSIBLE)
print(f"Bruto:       tantea {n_bruto:>6,} veces antes de rendirse y declarar 'imposible'.")
print(f"Propagando:  lo detecta tras {n_prop:>6} colocaciones (un dominio se queda vacio enseguida).")
print(f"\nMismo veredicto, pero propagar lo alcanza ~{n_bruto//max(n_prop,1)} veces antes.")
print("Descubrir pronto que una rama es imposible es media batalla de los CSP.")


## 13. El mismo esqueleto para otro problema: colorear un mapa

Aquí está la gracia del CSP: el método no sabe de sudokus. Si cambias variables, dominios y
restricciones, resuelve **otro** problema sin tocar la idea. Colorear un mapa con tres colores de modo
que dos regiones vecinas nunca compartan color es un CSP de manual:

- **Variables:** las regiones del mapa.
- **Dominios:** los colores disponibles (aquí, tres).
- **Restricciones:** regiones vecinas con colores distintos.

Usamos la topología clásica del mapa de Australia (siete regiones), un ejemplo típico de CSP. El
solucionador es el mismo de siempre —backtracking con MRV—, solo que escrito de forma genérica.


In [ ]:
# Mapa: 7 regiones (A-G). 'vecinas' lista, para cada una, con quien comparte frontera.
vecinas = {
    "A": {"B","C"},
    "B": {"A","C","D"},
    "C": {"A","B","D","E","F"},
    "D": {"B","C","E"},
    "E": {"C","D","F"},
    "F": {"C","E"},
    "G": set(),                       # una isla, sin fronteras
}
regiones = list("ABCDEFG")
colores  = ["rojo", "verde", "azul"]

def colorea_mapa(regiones, vecinas, colores):
    asign = {}; pasos = [0]
    def compatible(r, col):
        return all(asign.get(v) != col for v in vecinas[r])
    def libres(r):
        return [col for col in colores if compatible(r, col)]
    def backtrack():
        sin_color = [r for r in regiones if r not in asign]
        if not sin_color: return True
        r = min(sin_color, key=lambda x: len(libres(x)))    # MRV, igual que en el sudoku
        for col in libres(r):
            asign[r] = col; pasos[0] += 1
            if backtrack(): return True
            del asign[r]
        return False
    ok = backtrack()
    return asign, pasos[0], ok

asign, pasos, ok = colorea_mapa(regiones, vecinas, colores)
print(f"Resuelto: {ok}  en {pasos} colocaciones (las mismas ideas, otro problema).")
for r in regiones:
    print(f"  region {r}: {asign[r]}")
correcto = all(asign[r] != asign[v] for r in regiones for v in vecinas[r])
print(f"\nNinguna pareja de vecinas comparte color: {correcto}")


## 14. Pruébalo tú

1. **Quita la heurística MRV** en `resolver_csp` (`usar_mrv=False`) y deja el *forward checking*. ¿Sube
   mucho el número de colocaciones? Elegir *bien* la siguiente variable importa tanto como propagar.
2. **Quita el *forward checking*** (`usar_fc=False`): vuelves casi al backtracking bruto. Mide la
   diferencia con la celda de ablación.
3. **Endurece el mapa:** añade una región `"H"` vecina de A, C y E, o quítale un color a la lista. ¿Sigue
   teniendo solución? El mismo solucionador te lo dice.
4. **Tu propio sudoku:** sustituye `TABLERO` por uno que encuentres y vuelve a ejecutar. ¿Lo resuelven
   los *naked singles* solos, o hace falta conjeturar?
5. **Cambia el conflicto** de la sección 12 a una columna o una caja. ¿Sigue detectándolo pronto la
   propagación?


## 15. Errores comunes

- **Pensar que el backtracking es inútil.** Es la base de todo; lo que lo hace práctico es añadirle
  propagación y heurísticas. Solo, escala fatal.
- **Propagar pero olvidar deshacer.** Al retroceder hay que **restaurar** los dominios que tachaste, o
  el estado queda corrupto. Es el error de implementación más común en CSP.
- **Elegir mal la variable.** Sin MRV, exploras muchísimo más. La heurística de «la más restringida
  primero» es casi gratis y muy potente, como muestra la ablación.
- **Fiarse de un solo caso.** El método bruto a veces tiene suerte; por eso medimos varios sudokus y
  comparamos tendencias, no anécdotas.
- **Creer que esto es solo para sudokus.** El mismo método colorea mapas, cuadra horarios y configura
  productos. El sudoku es el *hola mundo* de los CSP.


## 16. Qué te llevas

- Un CSP son **variables, dominios y restricciones**. Modelar un problema así te da gratis un método
  general que sirve para muchísimos problemas reales (lo viste con el coloreado de mapas).
- **Propagar** las consecuencias de cada decisión (*forward checking*) y elegir primero **la variable
  más restringida** (MRV) convierte una búsqueda gigantesca en una manejable. La **ablación** te dice
  cuánto aporta cada ingrediente.
- A veces ni hace falta conjeturar: los **naked singles** resuelven los casos fáciles solo deduciendo.
- La ventaja **crece con la dificultad** y, sobre todo, es **estable**: propagar apenas retrocede,
  mientras el bruto es impredecible y descarta tarde lo imposible.
- La lección de fondo del facsímil: no basta con encontrar *una* solución tanteando; conviene razonar
  sobre las restricciones *antes* de actuar.

**Para seguir:** el capítulo 8 usa estas restricciones como *guardrails* de un sistema; los capítulos
9-10 saltan a planificación (PDDL), que es buscar en un espacio de estados con acciones; y la idea de
«propagar y podar» reaparece en casi todo lo que decide con reglas.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*